# Regime Strategy — Validation Backtest

Loads cached artifacts from `outputs/regime_cache/` (regressors, tech
features, regime classifier) produced by `MeanRev_vs_Momentum.ipynb`, then
runs the same PM logic on the **VALIDATION** window only. Test stays locked.

Requires `MeanRev_vs_Momentum.ipynb` to have been executed at least once.

## Set `RECOMPUTE_PM = True` to re-run PM with different gate params.

In [ ]:
# Cell 2 — Imports, cache, seeds
import os, sys, math, pickle, warnings
import random as _random
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists():
        PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

SEED = 42
_random.seed(SEED); np.random.seed(SEED)

CACHE_DIR = Path('outputs/regime_cache')
REGRESSOR_CACHE = CACHE_DIR / 'regressor_outputs.pkl'
TECH_CACHE      = CACHE_DIR / 'tech_features.pkl'
CLF_CACHE       = CACHE_DIR / 'regime_clf.pkl'

for p in [REGRESSOR_CACHE, TECH_CACHE, CLF_CACHE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing cache: {p}. Run MeanRev_vs_Momentum.ipynb first.')
print(f'Cache OK: {[p.name for p in [REGRESSOR_CACHE, TECH_CACHE, CLF_CACHE]]}')

In [ ]:
# Cell 3 — Project setup, splits
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split

cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)

train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(split.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(split.test_idx).sort_values()

TARGETS = {s['name']: {'etf': etf, 'target': s['target'], 'predictors': s['predictors']}
           for etf, s in SECTORS.items()}

print(f'VAL: {val_idx[0].date()} -> {val_idx[-1].date()} | n={len(val_idx)}')

In [ ]:
# Cell 4 — Load cached artifacts
with open(REGRESSOR_CACHE, 'rb') as f:
    regressor_outputs = pickle.load(f)
tech_panel = pd.read_pickle(TECH_CACHE)
with open(CLF_CACHE, 'rb') as f:
    clf_blob = pickle.load(f)
clf = clf_blob['model']
TECH_FEATURE_COLS = clf_blob['features']

print(f'Loaded {len(regressor_outputs)} sectors, '
      f'{len(tech_panel):,} tech rows, {len(TECH_FEATURE_COLS)} clf features')

In [ ]:
# Cell 5 — Build decision panel + restrict to VAL
_FROM_IDX = {0: -1, 1: 0, 2: 1}
panel_clf = tech_panel.copy()
panel_clf['date'] = pd.to_datetime(panel_clf['date'])
X_all = panel_clf[TECH_FEATURE_COLS].astype(float)
proba_all = clf.predict_proba(X_all)
panel_clf['regime_pred'] = np.array([_FROM_IDX[i] for i in clf.predict(X_all)])
panel_clf['P_MR']        = proba_all[:, 0]
panel_clf['P_NEUTRAL']   = proba_all[:, 1]
panel_clf['P_MOM']       = proba_all[:, 2]
panel_clf['regime_conf'] = proba_all.max(axis=1)

decision_rows = []
for sector, info in TARGETS.items():
    tgt = info['target']
    if sector not in regressor_outputs:
        continue
    reg = regressor_outputs[sector]
    clf_slice = panel_clf[panel_clf['sector'] == sector][
        ['date','regime_pred','regime_conf','P_MR','P_NEUTRAL','P_MOM']
    ].set_index('date')
    df = pd.DataFrame({
        'residual_z':       reg['residual_z'],
        'predicted_return': reg['predicted_return'],
        'next_ret':         reg['next_ret'],
        'price':            md.prices[tgt],
    }).join(clf_slice, how='left')
    df['sector'] = sector; df['target'] = tgt; df['date'] = df.index
    decision_rows.append(df.reset_index(drop=True))

decision_panel = pd.concat(decision_rows, ignore_index=True)
decision_panel = decision_panel.dropna(subset=['residual_z','regime_pred','next_ret']).reset_index(drop=True)
decision_panel = decision_panel.sort_values(['date','sector']).reset_index(drop=True)

val_panel = decision_panel[decision_panel['date'].isin(val_idx)].copy()
print(f'VAL decision panel: {len(val_panel):,} rows | '
      f'{val_panel["date"].min().date()} -> {val_panel["date"].max().date()} | '
      f'{val_panel["date"].nunique()} dates')
print('\nRegime distribution (VAL):')
display(val_panel['regime_pred'].map({-1:'MEAN_REV',0:'NEUTRAL',1:'MOMENTUM'}).value_counts())
print('\nresidual_z stats (VAL):')
display(val_panel['residual_z'].describe().round(3).to_frame('residual_z'))

In [ ]:
# Cell 6 — PM simulator (same logic as MeanRev notebook)
PM_COST_BPS = 5.0 / 1e4

def regime_pm_simulate(panel, rz_thr, conf_thr, mr_exit,
                       max_hold=10, stop_loss=-0.02, take_profit=0.03,
                       pred_ret_thr=0.001, cost_bps=PM_COST_BPS):
    state, seq, rows = {}, {}, []
    for (date, sector), grp in panel.groupby(['date','sector'], sort=True):
        r = grp.iloc[0]
        st = state.get(sector, dict(position=0, days_in=0, entry_price=np.nan,
                                    trade_pnl=0.0, trade_id=None))
        prev_pos = st['position']
        rz, regm, conf = float(r['residual_z']), int(r['regime_pred']), float(r['regime_conf'])
        prr  = float(r['predicted_return']) if pd.notna(r['predicted_return']) else 0.0
        nr   = float(r['next_ret']) if pd.notna(r['next_ret']) else 0.0
        price = float(r['price']) if pd.notna(r['price']) else np.nan
        action, desired = 'HOLD', prev_pos
        if prev_pos != 0:
            st['days_in'] += 1
            if (st['days_in'] >= max_hold or abs(rz) <= mr_exit
                or st['trade_pnl'] <= stop_loss or st['trade_pnl'] >= take_profit):
                desired = 0; action = 'EXIT'
        if desired == 0 and abs(rz) >= rz_thr and regm != 0 and conf >= conf_thr:
            if regm == -1:
                if rz < 0:  desired, action = 1, 'ENTER_LONG'
                else:       desired, action = -1, 'ENTER_SHORT'
            else:
                if rz > 0 and prr > -pred_ret_thr: desired, action = 1, 'ENTER_LONG'
                elif rz < 0 and prr < pred_ret_thr: desired, action = -1, 'ENTER_SHORT'
        turnover = abs(desired - prev_pos)
        gross = desired * nr; net = gross - turnover * cost_bps
        is_entry = action.startswith('ENTER'); is_exit = (action == 'EXIT')
        if is_entry:
            seq[sector] = seq.get(sector, 0) + 1
            st = dict(position=desired, days_in=1, entry_price=price,
                      trade_pnl=net, trade_id=seq[sector])
        elif is_exit:
            st['trade_pnl'] += net
            st = dict(position=0, days_in=0, entry_price=np.nan, trade_pnl=0.0, trade_id=None)
        elif desired != 0:
            st['trade_pnl'] += net
        state[sector] = st
        rows.append({
            'date': date, 'sector': sector, 'target': r['target'],
            'residual_z': rz, 'regime_pred': regm, 'regime_conf': conf,
            'predicted_return': prr, 'next_ret': nr,
            'action': action, 'position': float(desired), 'prev_pos': float(prev_pos),
            'turnover': float(turnover),
            'gross_pnl': float(gross), 'net_pnl': float(net),
            'is_entry': bool(is_entry), 'is_exit': bool(is_exit),
            'days_in_position': int(st['days_in']) if desired != 0 else 0,
        })
    return pd.DataFrame(rows)

def portfolio_metrics(td, periods_per_year=252):
    if td.empty: return pd.Series(dtype=float)
    daily = td.groupby('date')['net_pnl'].mean().sort_index()
    if daily.empty: return pd.Series(dtype=float)
    eq = (1.0 + daily).cumprod(); cum = float(eq.iloc[-1] - 1.0); n = len(daily)
    ann_ret = float((1.0 + cum) ** (periods_per_year / max(n,1)) - 1.0)
    ann_vol = float(daily.std(ddof=1) * np.sqrt(periods_per_year)) if n > 1 else np.nan
    sh = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol != 0 else np.nan
    dn = daily[daily < 0]
    dn_v = float(dn.std(ddof=1) * np.sqrt(periods_per_year)) if len(dn) > 1 else np.nan
    so = ann_ret / dn_v if dn_v and np.isfinite(dn_v) and dn_v != 0 else np.nan
    peak = eq.cummax(); mdd = float((eq / peak - 1.0).min())
    entries = td[td['is_entry']]
    return pd.Series({
        'trading_days': n, 'total_entries': int(len(entries)),
        'long_entries':  int((entries['position'] == 1).sum()),
        'short_entries': int((entries['position'] == -1).sum()),
        'active_days':   int((td['position'] != 0).sum()),
        'cumulative_return':     round(cum, 4),
        'annualized_return':     round(ann_ret, 4),
        'annualized_volatility': round(ann_vol, 4) if np.isfinite(ann_vol) else np.nan,
        'sharpe':                round(sh, 4) if np.isfinite(sh) else np.nan,
        'sortino':               round(so, 4) if np.isfinite(so) else np.nan,
        'max_drawdown':          round(mdd, 4),
        'win_rate_days':         round(float((daily > 0).mean()), 4),
        'avg_daily_pnl':         round(float(daily.mean()), 6),
    })

print('PM simulator + metrics function ready.')

In [ ]:
# Cell 7 — Baseline VAL run with TRAIN-tuned gates
RZ_THRESHOLD    = 1.5
REGIME_CONF_THR = 0.55
MR_EXIT         = 0.30

trades_val = regime_pm_simulate(val_panel, RZ_THRESHOLD, REGIME_CONF_THR, MR_EXIT)
metrics_val = portfolio_metrics(trades_val)
print('=' * 60)
print(f'VAL BASELINE (rz={RZ_THRESHOLD}, conf={REGIME_CONF_THR}, mre={MR_EXIT})')
print('=' * 60)
print(metrics_val.to_string())
print(f'\nAction distribution:\n{trades_val["action"].value_counts()}')

In [ ]:
# Cell 8 — Per-sector metrics on VAL
sec_rows = []
for sec, g in trades_val.groupby('sector'):
    m = portfolio_metrics(g)
    sec_rows.append({
        'sector': sec,
        'entries': int(m.get('total_entries', 0)),
        'sharpe':  m.get('sharpe', np.nan),
        'sortino': m.get('sortino', np.nan),
        'cum_ret': m.get('cumulative_return', np.nan),
        'max_dd':  m.get('max_drawdown', np.nan),
        'win_rate_days': m.get('win_rate_days', np.nan),
    })
display(pd.DataFrame(sec_rows).sort_values('sharpe', ascending=False, na_position='last'))

In [ ]:
# Cell 9 — Gate sweep on VAL
GRID = {
    'rz_thr':  [1.2, 1.5, 1.8, 2.0],
    'conf_thr': [0.45, 0.55, 0.65],
    'mr_exit': [0.20, 0.30, 0.50],
}
rows = []
combos = list(product(GRID['rz_thr'], GRID['conf_thr'], GRID['mr_exit']))
print(f'Sweeping {len(combos)} combos on VAL...')
for rz, c, mre in combos:
    td = regime_pm_simulate(val_panel, rz, c, mre)
    m = portfolio_metrics(td)
    rows.append({
        'rz_thr': rz, 'conf_thr': c, 'mr_exit': mre,
        **{k: m.get(k) for k in ['total_entries','long_entries','short_entries',
                                  'sharpe','sortino','cumulative_return',
                                  'annualized_return','annualized_volatility',
                                  'max_drawdown','win_rate_days','active_days']},
    })
    print(f'  rz={rz} conf={c} mre={mre} | '
          f'entries={int(m.get("total_entries",0)):3d} '
          f'sharpe={m.get("sharpe",np.nan):+.2f} '
          f'cumret={m.get("cumulative_return",np.nan):+.3f}')

sweep_df = pd.DataFrame(rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
sweep_df.to_csv('outputs/regime_cache/val_sweep.csv', index=False)
print(f'\nTop 10 by Sharpe:')
display(sweep_df.head(10))

In [ ]:
# Cell 10 — Log baseline + best sweep result to run_history
RUN_HISTORY = Path('outputs/tuning_cache/run_history.csv')
RUN_HISTORY.parent.mkdir(parents=True, exist_ok=True)
HIST_COLS = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
             'total_entries','sharpe','sortino','cumulative_return',
             'annualized_return','annualized_volatility','max_drawdown',
             'win_rate_days','active_days']
ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_rows = []
log_rows.append({
    'timestamp': ts, 'source': 'Regime_VAL_baseline',
    'tag': f'rz{RZ_THRESHOLD}_conf{REGIME_CONF_THR}_mre{MR_EXIT}_VAL',
    'clf_trial': None, 'f1_val': None,
    'rz_thr': RZ_THRESHOLD, 'conf': REGIME_CONF_THR, 'mr_exit': MR_EXIT,
    **{k: metrics_val.get(k) for k in HIST_COLS if k not in
       ('timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit')},
})
best = sweep_df.iloc[0]
log_rows.append({
    'timestamp': ts, 'source': 'Regime_VAL_sweep_best',
    'tag': f'rz{best["rz_thr"]}_conf{best["conf_thr"]}_mre{best["mr_exit"]}_VAL_BEST',
    'clf_trial': None, 'f1_val': None,
    'rz_thr': best['rz_thr'], 'conf': best['conf_thr'], 'mr_exit': best['mr_exit'],
    **{k: best.get(k) for k in HIST_COLS if k not in
       ('timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit')},
})
new_df = pd.DataFrame([{c: r.get(c) for c in HIST_COLS} for r in log_rows])
if RUN_HISTORY.exists():
    prev = pd.read_csv(RUN_HISTORY)
    combined = pd.concat([prev, new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(RUN_HISTORY, index=False)
print(f'Logged {len(new_df)} rows. Total in history: {len(combined)}')

print('\nAll Regime_* runs across train/val:')
display(combined[combined['source'].str.startswith('Regime')]
        .sort_values('sharpe', ascending=False))